# Сравнение детекции КП

Три подхода на одном изображении:

1. **CV** — классический пайплайн (`detect_controls`, `mode="circle_first"`): magenta-кольца → OCR.
2. **VLM raw** — Vision LLM (`detect_controls_vlm`, `refine_circles=False`): сырые координаты модели.
3. **VLM + refine** — те же VLM-точки после `_snap_to_ring` / `refine_ring_center` к magenta-кольцам.

VLM вызывается один раз; refine применяется поверх тех же детекций — честное сравнение snap.

Для VLM нужны `YC_API_KEY` и `YC_FOLDER_ID` в `.env` (см. `.env.example`).

In [ ]:
from pathlib import Path
import sys
import os

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("Не найден src/")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

In [ ]:
from __future__ import annotations

import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.controls import (
    animate_detect_controls_cv,
    animate_detect_controls_vlm,
    animate_detect_controls_vlm_refine,
    detect_controls,
    detect_controls_vlm_trace,
    draw_controls,
)
from src.controls.detect_vlm import _snap_to_ring, estimate_cp_radius_px
from src.controls.pink_mask import merge_prior, pink_mask_hsv
from src.controls.types import ControlPoint, CourseDetection

%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## 1. Входное изображение и размер кружка

Задайте `IMAGE` и **`CIRCLE_DIAMETER_PX`** (диаметр пурпурного кружка КП в пикселях *этого* кадра).
Параметр идёт и в CV (Hough), и в VLM refine (`_snap_to_ring`).


In [ ]:
# Карта / скрин для сравнения:
IMAGE = ROOT / "dataset" / "semenkino_2005_omaps" / "tiles" / "tile_0013" / "image.jpg"
# IMAGE = ROOT / "notebooks" / "assets" / "controls_compare_sample.png"  # превью ~1024×470
# IMAGE = Path("/path/to/your_1648x757.png")

# Диаметр кружка КП в пикселях ЭТОГО изображения (важно для CV Hough и VLM snap).
# Пример: кадр 1648×757 → Ø ≈ 60 px. На даунскейле пропорционально уменьшите.
CIRCLE_DIAMETER_PX = 60.0

PRIOR = IMAGE.parent / "channels.npy"
prior = np.load(PRIOR)[15] if PRIOR.exists() else None

assert IMAGE.exists(), IMAGE
rgb = np.array(Image.open(IMAGE).convert("RGB"))
h, w = rgb.shape[:2]
print(f"image: {IMAGE}")
print(f"shape: {w}×{h}  prior={'yes' if prior is not None else 'no'}")
print(f"CIRCLE_DIAMETER_PX={CIRCLE_DIAMETER_PX}  → radius≈{CIRCLE_DIAMETER_PX/2:.1f} px")

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(rgb)
ax.set_title(f"input  {w}×{h}  Ø={CIRCLE_DIAMETER_PX:.0f}px")
ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Детекция: CV / VLM raw / VLM + refine

In [ ]:
def refine_vlm_controls(
    rgb: np.ndarray,
    result: CourseDetection,
    *,
    prior_mask: np.ndarray | None = None,
    circle_diameter_px: float | None = 55.0,
) -> CourseDetection:
    """Применить `_snap_to_ring` + `refine_ring_center` к уже найденным VLM-точкам."""
    pink = pink_mask_hsv(
        rgb,
        use_clahe=True,
        min_saturation=25,
        min_value=30,
        preserve_thin=True,
    )
    pink = merge_prior(pink, prior_mask)
    expected_r = estimate_cp_radius_px(pink, hint_diameter=circle_diameter_px)

    refined: list[ControlPoint] = []
    for cp in result.controls:
        sx, sy, hollow, snapped = _snap_to_ring(
            pink, cp.x, cp.y, expected_radius=expected_r
        )
        if snapped:
            score = float(cp.score) * (0.55 + 0.45 * min(1.0, hollow))
            refined.append(
                ControlPoint(
                    number=cp.number,
                    x=float(sx),
                    y=float(sy),
                    score=score,
                    source="circle",
                )
            )
        else:
            refined.append(cp)

    return CourseDetection(controls=refined, start=result.start)


def summarize(name: str, det: CourseDetection) -> None:
    nums = sorted(c.number for c in det.controls)
    print(f"{name}: n={len(det.controls)}  numbers={nums}  start={det.start}")
    for c in det.controls:
        print(f"  {c}")

In [ ]:
# --- 1) классический CV ---
cv = detect_controls(
    rgb,
    prior_mask=prior,
    sensitivity="recall",
    min_ocr_conf=0.05,
    mode="circle_first",
    circle_diameter_px=CIRCLE_DIAMETER_PX,
)
summarize("CV", cv)


## 2.5. Анимация CV-детектора (circle-first)

Как `animate_path_search` в `route_demo`: GIF волны шагов на входном изображении.

Этапы: **маска → Hough-кольца → OCR-окна → принятые КП → итог**.
Сохраняется в `runs/controls_vis/` и сразу показывается в ячейке.


In [ ]:
# Анимация классического детектора (circle-first)
# Повторно прогоняет пайплайн с трассировкой — те же параметры, что у `cv` выше.

cv_anim, anim_path = animate_detect_controls_cv(
    rgb,
    prior_mask=prior,
    sensitivity="recall",
    min_ocr_conf=0.05,
    find_start=True,
    circle_diameter_px=CIRCLE_DIAMETER_PX,
    out_path=ROOT / "runs" / "controls_vis" / "cv_detect.gif",
    format="gif",  # или "mp4"
    fps=16,
    n_mask_frames=24,
    n_ring_frames=40,
    frames_per_ocr=6,
    hold_frames=28,
    max_side=1280,  # даунскейл длинной стороны (None = полный размер)
    display=True,
)

print(f"сохранено: {anim_path}")
print(f"КП: {sorted(c.number for c in cv_anim.controls)}")
print(f"start: {cv_anim.start}")


In [ ]:
# --- 2) VLM без refine (сырые координаты) ---
# Ключи: YC_API_KEY + YC_FOLDER_ID в .env
# При необходимости:
# import os
# os.environ["YC_API_KEY"] = "..."
# os.environ["YC_FOLDER_ID"] = "b1g..."

VLM_PROVIDER = "yandex"
VLM_MODEL = "gemma-3-27b-it"  # замените, если list_yandex_chat_models() покажет другую VL

vlm_trace = detect_controls_vlm_trace(
    rgb,
    provider=VLM_PROVIDER,
    model=VLM_MODEL,
    refine_circles=False,
    find_start=True,
    min_confidence=0.35,
    prior_mask=prior,
    circle_diameter_px=CIRCLE_DIAMETER_PX,
)
vlm_raw = vlm_trace.result
summarize("VLM raw", vlm_raw)


## 2.6. Анимация VLM без refine

Обход тайлов Vision LLM и появление сырых точек (без `refine_ring_center`).
Использует уже посчитанный `vlm_trace` — **без повторного API-вызова**.
GIF → `runs/controls_vis/vlm_raw.gif`.


In [ ]:
# Анимация VLM raw (тайлы → hits → итог), без refine
vlm_anim, vlm_anim_path = animate_detect_controls_vlm(
    rgb,
    trace=vlm_trace,  # тот же прогон, что дал vlm_raw
    out_path=ROOT / "runs" / "controls_vis" / "vlm_raw.gif",
    format="gif",  # или "mp4"
    fps=16,
    frames_per_tile=10,
    frames_per_hit=3,
    hold_frames=28,
    max_side=1280,
    display=True,
)

print(f"сохранено: {vlm_anim_path}")
print(f"КП: {sorted(c.number for c in vlm_anim.controls)}")
print(f"тайлов: {len(vlm_trace.tile_events)}")


In [ ]:
# --- 3) те же VLM-точки + refine_ring_center (через _snap_to_ring) ---
vlm_ref = refine_vlm_controls(
    rgb,
    vlm_raw,
    prior_mask=prior,
    circle_diameter_px=CIRCLE_DIAMETER_PX,
)
summarize("VLM + refine", vlm_ref)

## 2.7. Анимация VLM + refine

Поверх уже найденных VLM-точек: magenta-маска → `_snap_to_ring` / `refine_ring_center` → сдвиг к центру кольца.
API не вызывается. GIF → `runs/controls_vis/vlm_refine.gif`.


In [ ]:
# Анимация VLM + refine (raw → snap → итог), без повторного VLM-вызова
vlm_ref_anim, vlm_ref_anim_path = animate_detect_controls_vlm_refine(
    rgb,
    vlm_raw,
    prior_mask=prior,
    circle_diameter_px=CIRCLE_DIAMETER_PX,
    out_path=ROOT / "runs" / "controls_vis" / "vlm_refine.gif",
    format="gif",  # или "mp4"
    fps=16,
    n_mask_frames=20,
    frames_per_cp=8,
    hold_frames=28,
    max_side=1280,
    display=True,
)

print(f"сохранено: {vlm_ref_anim_path}")
print(f"КП: {sorted(c.number for c in vlm_ref_anim.controls)}")
n_snap = sum(1 for c in vlm_ref_anim.controls if c.source == "circle")
print(f"snap: {n_snap}/{len(vlm_ref_anim.controls)}")


## 3. Визуальное сравнение (side-by-side)

In [ ]:
METHODS = [
    ("CV (circle_first)", cv),
    ("VLM raw (без refine)", vlm_raw),
    ("VLM + refine_ring_center", vlm_ref),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (title, det) in zip(axes, METHODS):
    ax.imshow(draw_controls(rgb, det))
    ax.set_title(f"{title}\nn={len(det.controls)}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Оверлей трёх методов на одном кадре

Цвета: **синий** = CV, **оранжевый** = VLM raw, **зелёный** = VLM + refine.  
Стрелки raw → refine показывают сдвиг после `refine_ring_center`.

In [ ]:
COLOR = {
    "cv": (40 / 255, 120 / 255, 1.0),
    "vlm": (220 / 255, 140 / 255, 40 / 255),
    "ref": (0.0, 200 / 255, 80 / 255),
}


def _by_number(det: CourseDetection) -> dict[int, ControlPoint]:
    return {c.number: c for c in det.controls}


cv_map = _by_number(cv)
raw_map = _by_number(vlm_raw)
ref_map = _by_number(vlm_ref)
all_nums = sorted(set(cv_map) | set(raw_map) | set(ref_map))

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb)

for n in all_nums:
    if n in cv_map:
        c = cv_map[n]
        ax.plot(c.x, c.y, "o", ms=10, mfc="none", mec=COLOR["cv"], mew=2, zorder=3)
        ax.text(c.x + 8, c.y - 8, str(n), color=COLOR["cv"], fontsize=9, fontweight="bold")
    if n in raw_map:
        c = raw_map[n]
        ax.plot(c.x, c.y, "s", ms=8, mfc="none", mec=COLOR["vlm"], mew=2, zorder=3)
    if n in ref_map:
        c = ref_map[n]
        ax.plot(c.x, c.y, "^", ms=9, mfc="none", mec=COLOR["ref"], mew=2, zorder=3)
    if n in raw_map and n in ref_map:
        a, b = raw_map[n], ref_map[n]
        d = float(np.hypot(b.x - a.x, b.y - a.y))
        if d >= 0.5:
            ax.annotate(
                "",
                xy=(b.x, b.y),
                xytext=(a.x, a.y),
                arrowprops=dict(arrowstyle="->", color=COLOR["ref"], lw=1.2),
                zorder=2,
            )

ax.plot([], [], "o", mfc="none", mec=COLOR["cv"], mew=2, label="CV")
ax.plot([], [], "s", mfc="none", mec=COLOR["vlm"], mew=2, label="VLM raw")
ax.plot([], [], "^", mfc="none", mec=COLOR["ref"], mew=2, label="VLM + refine")
ax.legend(loc="upper right", framealpha=0.9)
ax.set_title("оверлей: CV / VLM raw / VLM + refine")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Кропы вокруг общих КП

Для номеров, найденных хотя бы двумя методами — зум, чтобы оценить точность центра кольца.

In [ ]:
CROP = max(70, int(round(CIRCLE_DIAMETER_PX * 1.2)))  # полуразмер кропа, px


def crop_window(img: np.ndarray, x: float, y: float, half: int) -> tuple[np.ndarray, int, int]:
    """Кроп вокруг (x,y) и origin кропа → локальные координаты: lx = X - x0."""
    h, w = img.shape[:2]
    cx, cy = int(round(x)), int(round(y))
    x0, y0 = max(0, cx - half), max(0, cy - half)
    x1, y1 = min(w, cx + half), min(h, cy + half)
    return img[y0:y1, x0:x1].copy(), x0, y0


shared = [
    n
    for n in all_nums
    if sum(n in m for m in (cv_map, raw_map, ref_map)) >= 2
]
if not shared:
    print("Нет общих номеров для кропов — смените IMAGE или пороги.")
else:
    n_show = min(len(shared), 8)
    fig, axes = plt.subplots(n_show, 3, figsize=(10, 3.2 * n_show))
    if n_show == 1:
        axes = np.array([axes])

    col_titles = ["CV", "VLM raw", "VLM + refine"]
    maps = [cv_map, raw_map, ref_map]
    colors = [COLOR["cv"], COLOR["vlm"], COLOR["ref"]]

    for row, n in enumerate(shared[:n_show]):
        pts = [m[n] for m in maps if n in m]
        cx = float(np.mean([p.x for p in pts]))
        cy = float(np.mean([p.y for p in pts]))
        patch, x0, y0 = crop_window(rgb, cx, cy, CROP)

        for col, (title, m, color) in enumerate(zip(col_titles, maps, colors)):
            ax = axes[row, col]
            ax.imshow(patch)
            if n in m:
                cp = m[n]
                ax.plot(cp.x - x0, cp.y - y0, "+", ms=14, mew=2, color=color)
                ax.set_title(f"#{n}  {title}\n({cp.x:.1f}, {cp.y:.1f})")
            else:
                ax.set_title(f"#{n}  {title}\n—")
            ax.axis("off")

    plt.tight_layout()
    plt.show()


## 6. Сводная таблица сдвигов

Для общих номеров: расстояние (px) между методами.

In [ ]:
def dist(a: ControlPoint | None, b: ControlPoint | None) -> float | None:
    if a is None or b is None:
        return None
    return float(np.hypot(a.x - b.x, a.y - b.y))


rows = []
for n in all_nums:
    a, b, c = cv_map.get(n), raw_map.get(n), ref_map.get(n)
    d_cv_raw = dist(a, b)
    d_cv_ref = dist(a, c)
    d_raw_ref = dist(b, c)
    rows.append(
        {
            "number": n,
            "cv": a is not None,
            "vlm_raw": b is not None,
            "vlm_ref": c is not None,
            "Δ cv↔raw": None if d_cv_raw is None else round(d_cv_raw, 1),
            "Δ cv↔ref": None if d_cv_ref is None else round(d_cv_ref, 1),
            "Δ raw→ref": None if d_raw_ref is None else round(d_raw_ref, 1),
            "ref_source": None if c is None else c.source,
        }
    )

try:
    import pandas as pd

    df = pd.DataFrame(rows)
    display(df)
    snapped = df["ref_source"].eq("circle").sum()
    print(f"snap сработал для {snapped}/{df['vlm_raw'].sum()} VLM-точек")
    if df["Δ raw→ref"].notna().any():
        print(
            "median |raw→ref| =",
            round(float(df["Δ raw→ref"].dropna().median()), 2),
            "px",
        )
except ImportError:
    for r in rows:
        print(r)